# ਸੈਮ੍ਰਾਂਟਿਕ ਕਰਨਲ

ਇਸ ਕੋਡ ਉਦਾਹਰਨ ਵਿੱਚ, ਤੁਸੀਂ ਇੱਕ ਸਰਲ ਏਜੰਟ ਬਣਾਉਣ ਲਈ [ਸੈਮ੍ਰਾਂਟਿਕ ਕਰਨਲ](https://aka.ms/ai-agents-beginners/semantic-kernel) AI ਫਰੇਮਵਰਕ ਦੀ ਵਰਤੋਂ ਕਰੋਗੇ।

ਇਸ ਉਦਾਹਰਨ ਦਾ ਉਦਦੇਸ਼ ਉਹ ਕਦਮ ਦਰਸਾਉਣਾ ਹੈ ਜੋ ਅਗਲੇ ਕੋਡ ਉਦਾਹਰਨਾਂ ਵਿੱਚ ਵੱਖ-ਵੱਖ ਏਜੰਟਿਕ ਪੈਟਰਨਾਂ ਨੂੰ ਲਾਗੂ ਕਰਨ ਵੇਲੇ ਵਰਤੇ ਜਾਣਗੇ।


## ਜਰੂਰੀ ਪਾਈਥਨ ਪੈਕੇਜ ਇੰਪੋਰਟ ਕਰੋ


In [ ]:
import json
import os 

from typing import Annotated

from dotenv import load_dotenv

from IPython.display import display, HTML

from openai import AsyncOpenAI

from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents import FunctionCallContent, FunctionResultContent, StreamingTextContent
from semantic_kernel.functions import kernel_function

## ਕਲਾਈਐਂਟ ਬਣਾਉਣਾ

ਇਸ ਉਦਾਹਰਣ ਵਿੱਚ, ਅਸੀਂ LLM ਤੱਕ ਪਹੁੰਚ ਲਈ [GitHub ਮਾਡਲ](https://aka.ms/ai-agents-beginners/github-models) ਦਾ ਇਸਤੇਮਾਲ ਕਰਾਂਗੇ।

`ai_model_id` ਨੂੰ `gpt-4o-mini` ਤੇ ਸੈੱਟ ਕੀਤਾ ਗਿਆ ਹੈ। ਵੱਖ-ਵੱਖ ਨਤੀਜੇ ਵੇਖਣ ਲਈ GitHub ਮਾਡਲਸ ਮਾਰਕੀਟਪਲੇਸ ਵਿੱਚ ਉਪਲਬਧ ਕਿਸੇ ਹੋਰ ਮਾਡਲ 'ਤੇ ਸਵਿੱਚ ਕਰਨ ਦੀ ਕੋਸ਼ਿਸ਼ ਕਰੋ।

GitHub ਮਾਡਲਸ ਲਈ ਲੋੜੀਂਦੇ `base_url` ਲਈ `Azure Inference SDK` ਵਰਤਣ ਲਈ, ਅਸੀਂ Semantic Kernel ਦੇ ਅੰਦਰ `OpenAIChatCompletion` ਕਨੈਕਟਰ ਦੀ ਵਰਤੋਂ ਕਰਾਂਗੇ। ਹੋਰ ਵੀ [ਉਪਲਬਧ ਕਨੈਕਟਰ](https://learn.microsoft.com/semantic-kernel/concepts/ai-services/chat-completion) ਹਨ ਜੋ ਤੁਹਾਨੂੰ ਵੱਖ-ਵੱਖ ਮਾਡਲ ਪ੍ਰਦਾਤਾਵਾਂ ਨਾਲ Semantic Kernel ਵਰਤਣ ਦੀ ਆਗਿਆ ਦਿੰਦੇ ਹਨ।


In [ ]:
import random   

# Define a sample plugin for the sample

class DestinationsPlugin:
    """A List of Random Destinations for a vacation."""

    def __init__(self):
        # List of vacation destinations
        self.destinations = [
            "Barcelona, Spain",
            "Paris, France",
            "Berlin, Germany",
            "Tokyo, Japan",
            "Sydney, Australia",
            "New York, USA",
            "Cairo, Egypt",
            "Cape Town, South Africa",
            "Rio de Janeiro, Brazil",
            "Bali, Indonesia"
        ]
        # Track last destination to avoid repeats
        self.last_destination = None

    @kernel_function(description="Provides a random vacation destination.")
    def get_random_destination(self) -> Annotated[str, "Returns a random vacation destination."]:
        # Get available destinations (excluding last one if possible)
        available_destinations = self.destinations.copy()
        if self.last_destination and len(available_destinations) > 1:
            available_destinations.remove(self.last_destination)

        # Select a random destination
        destination = random.choice(available_destinations)

        # Update the last destination
        self.last_destination = destination

        return destination

In [ ]:
load_dotenv()
client = AsyncOpenAI(
    api_key=os.environ.get("GITHUB_TOKEN"), 
    base_url="https://models.inference.ai.azure.com/",
)

# Create an AI Service that will be used by the `ChatCompletionAgent`
chat_completion_service = OpenAIChatCompletion(
    ai_model_id="gpt-4o-mini",
    async_client=client,
)

## ਏਜੰਟ ਬਣਾਉਣਾ

ਇੱਥੇ, ਅਸੀਂ `TravelAgent` ਨਾਮਕ ਏਜੰਟ ਬਣਾਉਂਦੇ ਹਾਂ।

ਇਸ ਉਦਾਹਰਨ ਵਿੱਚ, ਅਸੀਂ ਬਹੁਤ ਹੀ ਬੁਨਿਆਦੀ ਹਦਾਇਤਾਂ ਵਰਤਦੇ ਹਾਂ। ਤੁਸੀਂ ਆਜਾਦੀ ਨਾਲ ਇਹ ਹਦਾਇਤਾਂ ਬਦਲ ਸਕਦੇ ਹੋ ਤਾਂ ਜੋ ਏਜੰਟ ਦੇ ਵਿਹਾਰ ਵਿੱਚ ਕਿਵੇਂ ਤਬਦੀਲੀ ਆਉਂਦੀ ਹੈ, ਇਹ ਦੇਖ ਸਕੋ।


In [ ]:
agent = ChatCompletionAgent(
    service=chat_completion_service, 
    plugins=[DestinationsPlugin()],
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
)

## Running the Agents

ਹੁਣ ਅਸੀਂ `ChatHistory` ਸੈੱਟ ਕਰਕੇ ਅਤੇ ਉਸ ਵਿੱਚ `system_message` ਸ਼ਾਮਲ ਕਰਕੇ Agent ਨੂੰ ਚਲਾ ਸਕਦੇ ਹਾਂ। ਅਸੀਂ ਪਹਿਲਾਂ ਹੀ ਸਥਾਪਿਤ ਕੀਤੇ `AGENT_INSTRUCTIONS` ਦੀ ਵਰਤੋਂ ਕਰਾਂਗੇ।

ਇਹ ਸਭ ਸੈੱਟ ਕਰਨ ਤੋਂ ਬਾਦ, ਅਸੀਂ `user_inputs` ਬਣਾਉਂਦੇ ਹਾਂ, ਜੋ ਦਿਖਾਉਂਦੇ ਹਨ ਕਿ ਯੂਜ਼ਰ Agent ਨੂੰ ਕੀ ਭੇਜ ਰਿਹਾ ਹੈ। ਇਸ ਉਦਾਹਰਨ ਵਿੱਚ, ਸੁਨੇਹਾ `Plan me a sunny vacation` ਤੈਅ ਕੀਤਾ ਗਿਆ ਹੈ।

ਤੁਸੀਂ Agent ਦੇ ਵੱਖ-ਵੱਖ ਪ੍ਰਤੀਕਿਰਿਆ ਕਰਨ ਦੇ ਤਰੀਕੇ ਨੂੰ ਦੇਖਣ ਲਈ ਇਸ ਸੁਨੇਹੇ ਨੂੰ ਬਦਲ ਸਕਦੇ ਹੋ।


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]

async def main():
    thread: ChatHistoryAgentThread | None = None

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        agent_name = None
        full_response: list[str] = []
        function_calls: list[str] = []

        # Buffer to reconstruct streaming function call
        current_function_name = None
        argument_buffer = ""

        async for response in agent.invoke_stream(
            messages=user_input,
            thread=thread,
        ):
            thread = response.thread
            agent_name = response.name
            content_items = list(response.items)

            for item in content_items:
                if isinstance(item, FunctionCallContent):
                    if item.function_name:
                        current_function_name = item.function_name

                    # Accumulate arguments (streamed in chunks)
                    if isinstance(item.arguments, str):
                        argument_buffer += item.arguments
                elif isinstance(item, FunctionResultContent):
                    # Finalize any pending function call before showing result
                    if current_function_name:
                        formatted_args = argument_buffer.strip()
                        try:
                            parsed_args = json.loads(formatted_args)
                            formatted_args = json.dumps(parsed_args)
                        except Exception:
                            pass  # leave as raw string

                        function_calls.append(f"Calling function: {current_function_name}({formatted_args})")
                        current_function_name = None
                        argument_buffer = ""

                    function_calls.append(f"\nFunction Result:\n\n{item.result}")
                elif isinstance(item, StreamingTextContent) and item.text:
                    full_response.append(item.text)

        if function_calls:
            html_output += (
                "<div style='margin-bottom:10px'>"
                "<details>"
                "<summary style='cursor:pointer; font-weight:bold; color:#0066cc;'>Function Calls (click to expand)</summary>"
                "<div style='margin:10px; padding:10px; background-color:#f8f8f8; "
                "border:1px solid #ddd; border-radius:4px; white-space:pre-wrap; font-size:14px; color:#333;'>"
                f"{chr(10).join(function_calls)}"
                "</div></details></div>"
            )

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>{agent_name or 'Assistant'}:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))

await main()

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰੋਪੱਤਰ**:  
ਇਹ ਦਸਤਾਵੇਜ਼ AI ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਅਨੁਵਾਦਿਤ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸਹੀਤਾ ਲਈ ਕੋਸ਼ਿਸ਼ ਕਰਦੇ ਹਾਂ, ਕ੍ਰਿਪਾ ਜਾਣੋ ਕਿ ਆਟੋਮੈਟਿਕ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਣਸਹੀਤੀਆਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ اپنی ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਪ੍ਰਮਾਣਿਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਮਹੱਤਵਪੂਰਨ ਜਾਣਕਾਰੀ ਲਈ ਪ੍ਰੋਫੈਸ਼ਨਲ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫਾਰਿਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੀ ਵਰਤੋਂ ਨਾਲ ਪੈਦਾ ਹੋਣ ਵਾਲੀਆਂ ਬੁਝਾਰਤਾਂ ਜਾਂ ਗਲਤ ਫਹਿਮੀਆਂ ਲਈ ਜ਼ਿੰਮੇਵਾਰ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
